# 🚀 Pipeline Y — GPU Runner V3 (Google Colab T4)

**Pipeline:** Financial Recommendation System — Production Pipeline Y
**Model:** 04y V3 — TabNet SSL+MTL Income Sniper · **Hybrid View (15 human-friendly features)**
**Target:** Beat XGBoost baseline 0.8103 on `IncomeInvestment`

---

### ⚡ Pre-flight Checklist
1. **Runtime → Change runtime type → T4 GPU**
2. Sync your project to Google Drive (the folder must contain `Main_y/` and `Output/Pipeline_Y/`)
3. Run `01y_feature_engineering.py` **locally first** to generate Master Datasets
4. Make sure `Train_Master_X_NN.csv` and `Test_Master_X_NN.csv` are uploaded to `Output/Pipeline_Y/` on Drive

---

### 🔑 V3 vs V1 Architecture Changes
| | V1 | **V3 (this run)** |
|---|---|---|
| Input features | 30 (collinear) | **15 (Hybrid Base+Engineered view)** |
| Optuna CV folds | 3 | **5 (full stability)** |
| n\_steps | searched | **fixed = 3** |
| lambda\_sparse range | [1e-5, 1e-3] | **[1e-4, 1e-2] (relaxed)** |


In [ ]:
# ============================================================
# CELL 1 — Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('✅ Google Drive mounted at /content/drive')


In [1]:
# ============================================================
# CELL 2 — Configure Paths
# ============================================================
import os, sys

# ── STEP 1: Find your project folder on Drive ─────────────────────────────
# Run this helper to list what's in your Drive root and locate the project:
import os
drive_root = "/content/drive/MyDrive"
print("📂 Folders in your Drive root:")
for item in sorted(os.listdir(drive_root)):
    fpath = os.path.join(drive_root, item)
    if os.path.isdir(fpath):
        print(f"   📁 {item}")

print("\n👆 Find the folder that contains 'Main_y' and 'Dataset_Needs_SOTA.csv'")
print("   Then set PROJECT_ROOT below and re-run this cell.\n")

# ── STEP 2: SET THIS ──────────────────────────────────────────────────────
# Replace with the actual path you found above.
# Example: "/content/drive/MyDrive/Fintech_proj2/Buisness_cases_Fintech/BuisnessCase2"
PROJECT_ROOT = "/content/drive/MyDrive/BuisnessCase2"   # ← EDIT THIS

MAIN_X     = os.path.join(PROJECT_ROOT, "Main_y")
PIPELINE_X = os.path.join(PROJECT_ROOT, "Output", "Pipeline_Y")

# Add Main_y to Python path (needed by 04y to find utilsy.py)
if MAIN_X not in sys.path:
    sys.path.insert(0, MAIN_X)

# ── STEP 3: Validate ──────────────────────────────────────────────────────
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"MAIN_X       : {MAIN_X}")
print(f"PIPELINE_X   : {PIPELINE_X}")

required = {
    "Dataset_Needs_SOTA.csv":        os.path.join(PROJECT_ROOT, "Dataset_Needs_SOTA.csv"),
    "Train_Master_X_NN.csv":         os.path.join(PIPELINE_X, "Train_Master_X_NN.csv"),
    "Test_Master_X_NN.csv":          os.path.join(PIPELINE_X, "Test_Master_X_NN.csv"),
    "02y_performance_baseline.json": os.path.join(PIPELINE_X, "02y_performance_baseline.json"),
    "04y_train_tabnet_income.py":    os.path.join(MAIN_X, "04y_train_tabnet_income.py"),
    "utilsy.py":                     os.path.join(MAIN_X, "utilsy.py"),
}

print("\n📋 File check:")
all_ok = True
for name, path in required.items():
    ok = os.path.exists(path)
    print(f"  {'✅' if ok else '❌ MISSING'}  {name}")
    if not ok:
        all_ok = False

if not all_ok:
    print("\n⚠️  Fix PROJECT_ROOT or upload missing files, then re-run this cell.")
else:
    print("\n✅ All files present — ready to run Cell 3.")


📂 Folders in your Drive root:


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive'

In [ ]:
# ============================================================
# CELL 3 — Install Dependencies
# ============================================================
# Colab already has: torch, numpy, pandas, scikit-learn, matplotlib
# We only install what is missing.

print("📦 Installing missing dependencies...")
import subprocess, sys

pkgs = ["pytorch-tabnet", "optuna", "openpyxl", "xlrd", "lightgbm"]
for pkg in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "--quiet"], check=True)
    print(f"  ✅ {pkg}")

import torch
if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\n✅ GPU    : {gpu}")
    print(f"   VRAM  : {vram:.1f} GB")
    print(f"   CUDA  : {torch.version.cuda}")
else:
    raise RuntimeError("❌ No GPU — go to Runtime → Change runtime type → T4 GPU")

print("\n✅ Environment ready.")


In [ ]:
# ============================================================
# CELL 4 — Launch 04y_train_tabnet_income.py  (V3)
# ============================================================
# Runs the script directly in this Python process so that all
# output and tracebacks appear live in the cell output.
# ============================================================
import runpy, os, sys

script = os.path.join(MAIN_X, "04y_train_tabnet_income.py")
print(f"▶  Running: {script}\n{'='*60}")

# runpy executes the file as __main__ inside the current process
runpy.run_path(script, run_name="__main__")

print(f"\n{'='*60}\n✅ 04y complete — run Cell 5 to inspect results.")


In [ ]:
# ============================================================
# CELL 5 — Inspect Results & Artefacts
# ============================================================
import json, os
import pandas as pd
from IPython.display import Image, display

perf_path = os.path.join(PIPELINE_X, "04y_tabnet_performance.json")
if os.path.exists(perf_path):
    with open(perf_path) as f:
        perf = json.load(f)
    print("📊 04y TabNet V3 — Blind Test Set Performance")
    print("=" * 52)
    for target, metrics in perf.items():
        print(f"\n  {target}")
        for k, v in metrics.items():
            print(f"    {k:<20} {v}")
else:
    print("⚠️  04y_tabnet_performance.json not found — did Cell 4 complete?")

# List all 04y outputs
print(f"\n📁 04y outputs in {PIPELINE_X}:")
for fname in sorted(os.listdir(PIPELINE_X)):
    if fname.startswith("04y"):
        fpath = os.path.join(PIPELINE_X, fname)
        size  = os.path.getsize(fpath) / 1024
        print(f"   ✅  {fname:<48} {size:>7.1f} KB")

# Display saliency map
sm = os.path.join(PIPELINE_X, "04y_tabnet_saliency_map.png")
if os.path.exists(sm):
    print("\n🗺️  Attention Saliency Map:")
    display(Image(sm))


In [ ]:
# ============================================================
# CELL 6 — Pipeline Y Income Leaderboard
# ============================================================
import json, os
import pandas as pd

rows = [
    {"Model": "R&D XGB Optuna (7 feat)",      "AUC": 0.760,  "Note": "Baseline R&D"},
    {"Model": "R&D Keras MTL (7 feat)",        "AUC": 0.797,  "Note": "Neural MTL"},
    {"Model": "R&D TabNet SSL (7 feat)",       "AUC": 0.822,  "Note": "SSL+MTL R&D (06c)"},
]

# Load 02y baseline from JSON
b_path = os.path.join(PIPELINE_X, "02y_performance_baseline.json")
if os.path.exists(b_path):
    with open(b_path) as f:
        b = json.load(f)
    rows.append({"Model": "02y Giga-XGB Calibrated (30 feat)",
                 "AUC": b.get("IncomeInvestment", {}).get("AUC", "N/A"),
                 "Note": "Optuna + Isotonic Cal."})

# Load this run's TabNet result
t_path = os.path.join(PIPELINE_X, "04y_tabnet_performance.json")
if os.path.exists(t_path):
    with open(t_path) as f:
        t = json.load(f)
    rows.append({"Model": "04y TabNet V3 (Hybrid view, 15 feat)",
                 "AUC": t.get("IncomeInvestment", {}).get("AUC", "TBD"),
                 "Note": "Pipeline Y V2 ← THIS RUN"})

lb = pd.DataFrame(rows)
lb["AUC"] = pd.to_numeric(lb["AUC"], errors="coerce")
lb = lb.sort_values("AUC", ascending=False).reset_index(drop=True)

print("🏆 IncomeInvestment Leaderboard — Pipeline Y")
print("=" * 65)
print(lb.to_string(index=False))
best_row = lb[lb["Note"].str.contains("THIS RUN", na=False)]
if not best_row.empty:
    delta = float(best_row["AUC"].iloc[0]) - 0.8103
    print(f"\n  Δ vs XGB baseline   : {delta:+.4f}")
    print(f"  Δ vs R&D TabNet SSL : {float(best_row['AUC'].iloc[0])-0.822:+.4f}")
